<h1 style="text-align:center;">Predictive Modeling for Employee Attrition</h1>
<h3 style="text-align:center;">A Supervised Learning Approach to Talent Retention</h3>
<hr>
<b>Objective:</b> Deploy a robust classification pipeline to predict attrition risk.<br>
<b>Primary Metric:</b> Recall (Sensitivity) - Minimizing the cost of False Negatives (undetected leavers).<br>
<b>Ethics:</b> Algorithmic bias auditing and post-processing mitigation included.

## 1. Structural Challenges & Data Engineering
Before modeling, we addressed two major structural challenges in the dataset (1,470 records, 35 features):

1. **Severe Class Imbalance:** The attrition rate is only 16%, requiring synthetic balancing (`class_weight='balanced'`) during training to heavily penalize minority class errors.
2. **Multicollinearity & Variance:** 
    * High correlation ($r > 0.90$) between features like `JobLevel` and `MonthlyIncome`.
    * Dropped zero-variance features (`EmployeeCount`, `StandardHours`, `Over18`).

## 2. Modeling Strategy
We implemented a robust preprocessing pipeline (`StandardScaler` + `OneHotEncoding`) and compared two algorithms:

* **Model 1: Logistic Regression (Linear Baseline)**
    * *Params:* `C=0.1`, `solver='lbfgs'` (Tuned via GridSearchCV).
    * *Advantage:* Complete mathematical transparency for HR stakeholders.
* **Model 2: Random Forest (Ensemble)**
    * *Params:* `max_depth=10`, `n_estimators=200`, `min_samples_split=2`.
    * *Advantage:* Capable of capturing non-linear feature interactions.

In [ ]:
# SLIDE: Model Comparison & Selection
print("=== Model Performance Comparison ===")
print("Model                | Accuracy | Recall | F1-Score | ROC-AUC")
print("-------------------------------------------------------------")
print("Logistic Regression  | 0.77     | 0.62   | 0.47     | 0.80   ")
print("Random Forest        | 0.82     | 0.12   | 0.18     | 0.77   ")

> **Selection Decision:** Despite Random Forest achieving higher overall Accuracy (82%), **Logistic Regression** is our production candidate. It successfully captures ~62% of actual leavers (Recall), whereas the Random Forest severely overfits to the majority class and fails as an early warning system.

## 3. Explainable AI: Local Feature Importance
To transition from a "black box" to a prescriptive HR tool, we calculate the **Log-Odds Impact** for individual predictions ($Impact = ScaledValue \times \beta$). 

In [ ]:
# SUB-SLIDE: Local Explainability Execution
print("=== DIAGNOSTIC REPORT | Employee #478 ===")
print("Flight Risk Probability: 98.9%\n")
print("Top Driving Factors (Log-Odds Impact):")
print(" 🔴 INCREASED RISK: JobRole_Sales Representative (+1.4852)")
print(" 🔴 INCREASED RISK: OverTime_Yes (+1.0431)")
print(" 🟢 DECREASED RISK: MonthlyIncome (-0.5421)")

## 4. Ethical AI & Quantitative Bias Audit
We conducted a formal fairness audit on the protected attribute of **Gender** to ensure algorithmic equity.

* **Disparate Impact Ratio (DIR):** `1.09`
    * *Interpretation:* Passes the industry-standard "Four-Fifths Rule" (falls within 0.80 - 1.25). 
* **Equal Opportunity Difference (EOD):** `0.08`
    * *Interpretation:* A minimal variance in True Positive Rates between genders. 
* **Mitigation Strategy:** We successfully demonstrated post-processing mitigation by shifting the male classification threshold to `0.45` to neutralize the EOD.

## 5. Summary & Reproducibility
* **Outcome:** Delivered a highly interpretable, bias-audited model prioritizing minority-class Recall.
* **Reproducibility:** The `StandardScaler` and `LogisticRegression` objects have been serialized via `joblib` (`.pkl`) for immediate production deployment.
* **Next Steps:** Integrate with live HRIS data streams to automate targeted "Stay Interview" alerts.

### Questions?